# Notebook 7 — Edge-AI Optimization

**Methods:** TorchScript · Dynamic Quantization (INT8) · Weight Pruning · Latency Benchmarking  
**Goal:** Make ECGRiskNet-XAI deployable on edge/mobile devices (Raspberry Pi, Jetson Nano, etc.)

## What changed from old NB7
- Model is now `ECGRiskNetXAI` with `(ecg, meta)` dual-input forward signature
- TorchScript uses `torch.jit.trace` with both ECG and meta dummy tensors
- Latency measured for single-sample inference (realistic edge scenario)
- Accuracy preservation check after each optimization step


In [1]:
import torch, sys, json, time, warnings
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")

device   = torch.device("cpu")   # Edge optimization always on CPU
SAVE_DIR = Path("./ECG_Project/data")
sys.path.insert(0, str(SAVE_DIR))
from ecg_model import ECGRiskNetXAI

with open(SAVE_DIR / "metadata.json") as f: META = json.load(f)
RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}

# Load model to CPU
model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3)
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", map_location="cpu"))
model.eval()
print("✅ Model loaded for edge optimization (CPU mode).")

def model_size_mb(m):
    total = sum(p.numel() for p in m.parameters())
    return total * 4 / 1e6   # float32 = 4 bytes

print(f"Original model size : {model_size_mb(model):.2f} MB")


✅ Model loaded for edge optimization (CPU mode).
Original model size : 20.21 MB


## Step 1: Baseline Latency

In [2]:
# Dummy inputs: single sample (batch_size=1)
dummy_ecg  = torch.randn(1, 12, 1000)
dummy_meta = torch.randn(1, 3)


def measure_latency(m, ecg_x, meta_x, n=100, is_scripted=False):
    """
    Measure average inference latency over n runs.
    Handles both ECGRiskNetXAI (dict output) and scripted models.
    """
    m.eval()
    with torch.no_grad():
        for _ in range(10):   # warm-up
            if is_scripted:
                m(ecg_x, meta_x)
            else:
                m(ecg_x, meta_x)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n):
            if is_scripted:
                m(ecg_x, meta_x)
            else:
                m(ecg_x, meta_x)
    return (time.perf_counter() - t0) / n * 1000   # ms per sample


baseline_lat = measure_latency(model, dummy_ecg, dummy_meta)
baseline_sz  = model_size_mb(model)
print(f"Baseline latency : {baseline_lat:.2f} ms/sample")
print(f"Baseline size    : {baseline_sz:.2f} MB")


Baseline latency : 13.17 ms/sample
Baseline size    : 20.21 MB


## Step 2: Dynamic Quantization (INT8)

In [3]:
import torch.quantization as tq

# Dynamic quantization on Linear layers (works without calibration data)
model_quant = tq.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
model_quant.eval()

quant_lat = measure_latency(model_quant, dummy_ecg, dummy_meta)
quant_sz  = model_size_mb(model_quant)   # approximate (quantized params stored differently)

print(f"Quantized latency : {quant_lat:.2f} ms/sample")
print(f"Speedup           : {baseline_lat / quant_lat:.2f}x")

# Save quantized model state dict only (scripted save not always reliable for quantized)
try:
    torch.save(model_quant.state_dict(), SAVE_DIR / "model_quantized.pt")
    print("✅ Quantized model saved.")
except Exception as e:
    print(f"Quantized save warning: {e}")


Quantized latency : 15.04 ms/sample
Speedup           : 0.88x
✅ Quantized model saved.


## Step 3: Weight Pruning (L1 Unstructured, 20% sparsity)

In [4]:
import torch.nn.utils.prune as prune

model_pruned = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3)
model_pruned.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", map_location="cpu"))
model_pruned.eval()

# Apply L1 unstructured pruning to all Linear layers
for name, module in model_pruned.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.20)
        prune.remove(module, "weight")   # make permanent (removes mask)

pruned_lat = measure_latency(model_pruned, dummy_ecg, dummy_meta)
pruned_sz  = model_size_mb(model_pruned)

# Count actual sparsity
total, zeros = 0, 0
for p in model_pruned.parameters():
    total += p.numel()
    zeros += (p == 0).sum().item()

print(f"Pruned latency  : {pruned_lat:.2f} ms/sample")
print(f"Pruned size     : {pruned_sz:.2f} MB")
print(f"Sparsity        : {100*zeros/total:.1f}%")
print(f"Speedup vs baseline: {baseline_lat / pruned_lat:.2f}x")

torch.save(model_pruned.state_dict(), SAVE_DIR / "model_pruned.pt")
print("✅ Pruned model saved.")


Pruned latency  : 13.35 ms/sample
Pruned size     : 20.21 MB
Sparsity        : 10.4%
Speedup vs baseline: 0.99x
✅ Pruned model saved.


## Step 4: TorchScript Export

TorchScript is the PyTorch equivalent of TFLite — runs without Python interpreter.  
Uses `torch.jit.trace` with both ECG and meta dummy tensors (dual-input model).


In [6]:
class ECGRiskNetWrapper(torch.nn.Module):
    """
    Thin wrapper that returns only the risk logits tensor (not a dict).
    TorchScript export needs a single tensor output for deployment.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ecg: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:
        out = self.model(ecg, meta)
        return out["risk"]   # (B, 4)


try:
    wrapper = ECGRiskNetWrapper(model)
    wrapper.eval()
    try:
        scripted = torch.jit.script(wrapper)
    except Exception as script_err:
        print(f"TorchScript scripting failed, falling back to tracing: {script_err}")
        scripted = torch.jit.trace(
            wrapper,
            (dummy_ecg, dummy_meta),
            check_trace=False,
            strict=False,
        )

    scripted.save(str(SAVE_DIR / "model_scripted.pt"))
    script_lat = measure_latency(scripted, dummy_ecg, dummy_meta, is_scripted=True)
    print(f"TorchScript latency : {script_lat:.2f} ms/sample")
    print(f"Speedup vs baseline : {baseline_lat / script_lat:.2f}x")
    print("✅ TorchScript model saved.")
except Exception as e:
    print(f"TorchScript export warning: {e}")
    scripted = wrapper
    script_lat = baseline_lat


TorchScript latency : 11.06 ms/sample
Speedup vs baseline : 1.19x
✅ TorchScript model saved.


## Step 5: Accuracy Preservation Check

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

test_d = np.load(SAVE_DIR / "test_processed.npz")
X_test = test_d["X"].astype(np.float32)
y_test = test_d["y"].astype(np.int64)
M_test = test_d["meta"].astype(np.float32) if "meta" in test_d \
         else np.zeros((len(y_test), 3), np.float32)

X_t = torch.from_numpy(X_test.transpose(0, 2, 1))   # (N, 12, 1000)
M_t = torch.from_numpy(M_test)
loader = DataLoader(TensorDataset(X_t, M_t), batch_size=64, shuffle=False, num_workers=0)


def quick_eval(m, loader, use_wrapper=False):
    m.eval()
    preds = []
    with torch.no_grad():
        for xb, mb in loader:
            if use_wrapper:
                logits = m(xb, mb)
            else:
                out    = m(xb, mb)
                logits = out["risk"]
            preds.extend(logits.argmax(1).numpy())
    preds = np.array(preds)
    acc = (preds == y_test).mean()
    mf1 = f1_score(y_test, preds, average="macro", zero_division=0)
    return float(acc), float(mf1)


print("Accuracy preservation check on test set:")
acc_orig,   mf1_orig   = quick_eval(model,        loader)
acc_quant,  mf1_quant  = quick_eval(model_quant,  loader)
acc_pruned, mf1_pruned = quick_eval(model_pruned,  loader)
acc_script, mf1_script = quick_eval(scripted,     loader, use_wrapper=True)

print(f"  Original    : Acc={acc_orig*100:.2f}%  Macro F1={mf1_orig:.4f}")
print(f"  Quantized   : Acc={acc_quant*100:.2f}%  Macro F1={mf1_quant:.4f}  "
      f"(drop={abs(acc_orig-acc_quant)*100:.2f}%)")
print(f"  Pruned      : Acc={acc_pruned*100:.2f}%  Macro F1={mf1_pruned:.4f}  "
      f"(drop={abs(acc_orig-acc_pruned)*100:.2f}%)")
print(f"  TorchScript : Acc={acc_script*100:.2f}%  Macro F1={mf1_script:.4f}  "
      f"(drop={abs(acc_orig-acc_script)*100:.2f}%)")


## Step 6: Edge Metrics Summary & Visualization

In [ ]:
metrics = {
    "Original"   : {"latency_ms": baseline_lat,  "size_mb": baseline_sz,
                    "accuracy": acc_orig,   "macro_f1": mf1_orig},
    "Quantized"  : {"latency_ms": quant_lat,      "size_mb": quant_sz,
                    "accuracy": acc_quant,  "macro_f1": mf1_quant},
    "Pruned"     : {"latency_ms": pruned_lat,     "size_mb": pruned_sz,
                    "accuracy": acc_pruned, "macro_f1": mf1_pruned},
    "TorchScript": {"latency_ms": script_lat,     "size_mb": baseline_sz,
                    "accuracy": acc_script, "macro_f1": mf1_script},
}

print(f"\n{'Method':>12} {'Latency (ms)':>14} {'Size (MB)':>10} "
      f"{'Speedup':>9} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 70)
for name, m in metrics.items():
    sp = baseline_lat / m["latency_ms"]
    print(f"{name:>12} {m['latency_ms']:>14.2f} {m['size_mb']:>10.2f} "
          f"{sp:>9.2f}x {m['accuracy']*100:>9.2f}% {m['macro_f1']:>10.4f}")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
names  = list(metrics.keys())
lats   = [metrics[n]["latency_ms"] for n in names]
sizes  = [metrics[n]["size_mb"]    for n in names]
f1s    = [metrics[n]["macro_f1"]   for n in names]
colors = ["steelblue", "orange", "green", "purple"]

axes[0].bar(names, lats,  color=colors)
axes[0].set_title("Inference Latency (ms)", fontweight="bold")
axes[0].set_ylabel("ms/sample"); axes[0].grid(True, alpha=0.3)

axes[1].bar(names, sizes, color=colors)
axes[1].set_title("Model Size (MB)", fontweight="bold")
axes[1].set_ylabel("MB"); axes[1].grid(True, alpha=0.3)

axes[2].bar(names, f1s,  color=colors)
axes[2].set_title("Macro F1 Score (Test)", fontweight="bold")
axes[2].set_ylabel("Macro F1"); axes[2].set_ylim(0, 1.1)
axes[2].grid(True, alpha=0.3)
for bar, val in zip(axes[2].patches, f1s):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontsize=9)

plt.suptitle("Edge-AI Optimization — ECGRiskNet-XAI", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "edge_optimization.png", dpi=150)
plt.show()

# Save metrics (drop numpy arrays)
metrics_json = {k: {mk: float(mv) for mk, mv in v.items()}
                for k, v in metrics.items()}
with open(SAVE_DIR / "edge_metrics.json", "w") as f:
    json.dump(metrics_json, f, indent=2)

print("\nNOTEBOOK 7 COMPLETE ✅")
print(f"  {SAVE_DIR}/model_quantized.pt")
print(f"  {SAVE_DIR}/model_pruned.pt")
print(f"  {SAVE_DIR}/model_scripted.pt")
print(f"  {SAVE_DIR}/edge_metrics.json")
print(f"  {SAVE_DIR}/edge_optimization.png")
print("Next → Notebook 8: 4-Tier Final Inference Pipeline & Clinical Report")
